In [0]:
# Run in a Databricks notebook cell to install required libraries
# OR add via Cluster → Libraries → Install New

%pip install faker==24.0.0
%pip install confluent-kafka==2.3.0
%pip install boto3==1.34.0

# Restart Python after pip installs
%restart_python

In [0]:

# REALISTIC FINTECH TRANSACTION DATA GENERATOR (DATABRICKS EDITION)
# Generates synthetic data with embedded fraud patterns directly to S3


import random
import uuid
import math
from datetime import datetime, timedelta
from faker import Faker
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Tuple

fake = Faker(['en_IN', 'en_US'])   # Indian + US locales
random.seed(42)


# DATA SCHEMAS


@dataclass
class CustomerProfile:
    customer_id: str
    full_name: str
    email: str
    phone: str
    date_of_birth: str
    kyc_status: str           
    account_status: str       
    risk_score: float         
    account_age_days: int
    country: str
    city: str
    state: str
    pin_code: str
    income_band: str          
    device_fingerprint: str
    registered_devices: List[str]
    preferred_payment: str
    typical_transaction_amount: float
    typical_merchant_categories: List[str]
    is_politically_exposed: bool
    account_open_date: str
    last_login_date: str
    failed_login_count: int
    is_synthetic_fraud_customer: bool = False
    fraud_type: Optional[str] = None


@dataclass
class MerchantProfile:
    merchant_id: str
    merchant_name: str
    merchant_category: str    
    merchant_category_code: str
    business_type: str        
    country: str
    city: str
    state: str
    latitude: float
    longitude: float
    risk_category: str        
    historical_fraud_score: float
    avg_transaction_amount: float
    monthly_transaction_volume: int
    accepts_international: bool
    is_high_risk_category: bool
    merchant_tenure_days: int
    chargeback_rate: float    
    registered_date: str


@dataclass
class Transaction:
    transaction_id: str
    customer_id: str
    merchant_id: str
    amount: float
    currency: str
    timestamp: str
    transaction_date: str     
    location_city: str
    location_state: str
    location_country: str
    latitude: float
    longitude: float
    ip_address: str
    device_id: str
    device_type: str          
    payment_method: str       
    transaction_status: str   
    merchant_category: str
    is_foreign_transaction: bool
    failed_attempt_count: int
    response_code: str
    hour_of_day: int
    day_of_week: int
    is_weekend: bool
    is_night_transaction: bool  
    is_fraud: bool = False
    fraud_type: Optional[str] = None
    fraud_score: float = 0.0



# REFERENCE DATA


INDIAN_CITIES = [
    ("Mumbai",    "Maharashtra",  19.0760,  72.8777),
    ("Delhi",     "Delhi",        28.6139,  77.2090),
    ("Bangalore", "Karnataka",    12.9716,  77.5946),
    ("Hyderabad", "Telangana",    17.3850,  78.4867),
    ("Chennai",   "Tamil Nadu",   13.0827,  80.2707),
    ("Kolkata",   "West Bengal",  22.5726,  88.3639),
    ("Pune",      "Maharashtra",  18.5204,  73.8567),
    ("Ahmedabad", "Gujarat",      23.0225,  72.5714),
    ("Jaipur",    "Rajasthan",    26.9124,  75.7873),
    ("Surat",     "Gujarat",      21.1702,  72.8311),
    ("Lucknow",   "UP",           26.8467,  80.9462),
    ("Kochi",     "Kerala",        9.9312,  76.2673),
    ("Chandigarh","Punjab",       30.7333,  76.7794),
    ("Bhopal",    "MP",           23.2599,  77.4126),
    ("Nagpur",    "Maharashtra",  21.1458,  79.0882),
    ("Jammu",     "J&K",          32.7266,  74.8570),
    ("Amritsar",  "Punjab",       31.6340,  74.8723),
]

FOREIGN_LOCATIONS = [
    ("Dubai",      "Dubai",     "AE",  25.2048,  55.2708),
    ("Singapore",  "Singapore", "SG",   1.3521, 103.8198),
    ("Hong Kong",  "HK",        "HK",  22.3193, 114.1694),
    ("London",     "England",   "GB",  51.5074,  -0.1278),
    ("New York",   "NY",        "US",  40.7128, -74.0060),
    ("Lagos",      "Lagos",     "NG",   6.5244,   3.3792), 
    ("Nairobi",    "Nairobi",   "KE",  -1.2921,  36.8219), 
]

MERCHANT_CATEGORIES = [
    ("GROCERY",         "5411", False, 0.02),
    ("RESTAURANT",      "5812", False, 0.03),
    ("FUEL",            "5541", False, 0.04),
    ("ELECTRONICS",     "5732", True,  0.08),
    ("JEWELRY",         "5944", True,  0.15),
    ("ONLINE_GAMING",   "7993", True,  0.20),
    ("CRYPTO_EXCHANGE", "6051", True,  0.35),
    ("MONEY_TRANSFER",  "4829", True,  0.25),
    ("LUXURY_GOODS",    "5999", True,  0.18),
    ("TRAVEL",          "4722", False, 0.06),
    ("HEALTHCARE",      "8099", False, 0.02),
    ("EDUCATION",       "8299", False, 0.01),
    ("UTILITY",         "4900", False, 0.01),
    ("ENTERTAINMENT",   "7922", False, 0.05),
    ("FASHION",         "5699", False, 0.04),
    ("TELECOM",         "4813", False, 0.03),
]

PAYMENT_METHODS = ["UPI", "CREDIT_CARD", "DEBIT_CARD", "NETBANKING", "WALLET", "BNPL"]
DEVICE_TYPES    = ["MOBILE", "DESKTOP", "POS", "ATM", "TABLET"]
CURRENCIES      = ["INR"] * 80 + ["USD", "EUR", "GBP", "AED", "SGD"] * 4


# CUSTOMER GENERATOR


class CustomerGenerator:
    def __init__(self, num_customers: int = 1000):
        self.num_customers = num_customers
        self.customers: List[CustomerProfile] = []

    def _generate_risk_score(self, kyc: str, account_age: int, income: str, pep: bool) -> float:
        score = 0.0
        if kyc == "PENDING":  score += 0.2
        if kyc == "FAILED":   score += 0.5
        if account_age < 30:  score += 0.3
        if income == "LOW":   score += 0.05
        if pep:               score += 0.25
        score += random.gauss(0, 0.05)
        return min(max(score, 0.0), 1.0)

    def generate(self) -> List[CustomerProfile]:
        for i in range(self.num_customers):
            city_data  = random.choice(INDIAN_CITIES)
            account_age = random.randint(1, 2000)
            kyc        = random.choices(["VERIFIED", "PENDING", "FAILED"], weights=[75, 20, 5])[0]
            income = random.choices(["LOW", "MEDIUM", "HIGH", "ULTRA_HIGH"], weights=[30, 45, 20, 5])[0]
            pep = random.random() < 0.02

            income_amounts = {
                "LOW":        (50, 2000),
                "MEDIUM":     (500, 15000),
                "HIGH":       (2000, 100000),
                "ULTRA_HIGH": (10000, 500000),
            }
            lo, hi = income_amounts[income]
            typical_amount = round(random.uniform(lo, hi), 2)

            open_date = datetime.now() - timedelta(days=account_age)
            last_login = datetime.now() - timedelta(days=random.randint(0, min(account_age, 30)))

            is_fraud_customer = (i % 5 == 0)
            fraud_type = None
            if is_fraud_customer:
                fraud_type = random.choice([
                    "ACCOUNT_TAKEOVER", "SYNTHETIC_IDENTITY", "CARD_NOT_PRESENT", 
                    "AML_STRUCTURING", "VELOCITY_ABUSE"
                ])

            num_devices = random.randint(1, 4 if is_fraud_customer else 2)
            devices = [str(uuid.uuid4())[:12] for _ in range(num_devices)]

            customer = CustomerProfile(
                customer_id                = f"CUST_{str(uuid.uuid4())[:8].upper()}",
                full_name                  = fake.name(),
                email                      = fake.email(),
                phone                      = f"+91{random.randint(6000000000, 9999999999)}",
                date_of_birth              = fake.date_of_birth(minimum_age=18, maximum_age=75).isoformat(),
                kyc_status                 = kyc,
                account_status             = random.choices(["ACTIVE","SUSPENDED","UNDER_REVIEW"], weights=[85, 10, 5])[0],
                risk_score                 = self._generate_risk_score(kyc, account_age, income, pep),
                account_age_days           = account_age,
                country                    = "IN",
                city                       = city_data[0],
                state                      = city_data[1],
                pin_code                   = str(random.randint(100000, 999999)),
                income_band                = income,
                device_fingerprint         = str(uuid.uuid4())[:16],
                registered_devices         = devices,
                preferred_payment          = random.choice(PAYMENT_METHODS),
                typical_transaction_amount = typical_amount,
                typical_merchant_categories= random.sample([m[0] for m in MERCHANT_CATEGORIES], k=3),
                is_politically_exposed     = pep,
                account_open_date          = open_date.date().isoformat(),
                last_login_date            = last_login.date().isoformat(),
                failed_login_count         = random.randint(0, 10 if is_fraud_customer else 2),
                is_synthetic_fraud_customer= is_fraud_customer,
                fraud_type                 = fraud_type,
            )
            self.customers.append(customer)

        print(f"✅ Generated {len(self.customers)} customers ({sum(1 for c in self.customers if c.is_synthetic_fraud_customer)} fraud-prone)")
        return self.customers

    def save_to_s3(self, output_path: str, spark):
        """Save customers directly to S3 via Spark DataFrames"""
        filepath = f"{output_path}/customers_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        rows = []
        for c in self.customers:
            row = asdict(c)
            row['registered_devices'] = "|".join(c.registered_devices)
            row['typical_merchant_categories'] = "|".join(c.typical_merchant_categories)
            rows.append(row)

        df = spark.createDataFrame(rows)
        # Coalesce into 1 file for clean CSV output in the lake
        df.coalesce(1).write.mode("overwrite").option("header", "true").csv(filepath)
        print(f"✅ Customers saved directly to S3 → {filepath}")
        return filepath



# MERCHANT GENERATOR


class MerchantGenerator:
    def __init__(self, num_merchants: int = 200):
        self.num_merchants = num_merchants
        self.merchants: List[MerchantProfile] = []

    def generate(self) -> List[MerchantProfile]:
        for _ in range(self.num_merchants):
            cat_data     = random.choice(MERCHANT_CATEGORIES)
            cat_name, mcc, is_high_risk, base_fraud_score = cat_data
            city_data    = random.choice(INDIAN_CITIES)
            tenure_days  = random.randint(30, 3650)

            fraud_score = min(base_fraud_score + random.gauss(0, 0.05), 1.0)

            merchant = MerchantProfile(
                merchant_id               = f"MERCH_{str(uuid.uuid4())[:8].upper()}",
                merchant_name             = fake.company(),
                merchant_category         = cat_name,
                merchant_category_code    = mcc,
                business_type             = random.choice(["ONLINE","PHYSICAL","HYBRID"]),
                country                   = "IN",
                city                      = city_data[0],
                state                     = city_data[1],
                latitude                  = city_data[2] + random.gauss(0, 0.05),
                longitude                 = city_data[3] + random.gauss(0, 0.05),
                risk_category             = ("VERY_HIGH" if fraud_score > 0.7 else "HIGH" if fraud_score > 0.4 else "MEDIUM" if fraud_score > 0.2 else "LOW"),
                historical_fraud_score    = round(fraud_score, 4),
                avg_transaction_amount    = round(random.uniform(100, 50000), 2),
                monthly_transaction_volume= random.randint(50, 50000),
                accepts_international     = random.random() < 0.3,
                is_high_risk_category     = is_high_risk,
                merchant_tenure_days      = tenure_days,
                chargeback_rate           = round(base_fraud_score * random.uniform(0.5, 1.5), 4),
                registered_date           = (datetime.now() - timedelta(days=tenure_days)).date().isoformat(),
            )
            self.merchants.append(merchant)

        print(f"✅ Generated {len(self.merchants)} merchants")
        return self.merchants

    def save_to_s3(self, output_path: str, spark):
        """Save merchants directly to S3 via Spark DataFrames"""
        filepath = f"{output_path}/merchants_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        rows = [asdict(m) for m in self.merchants]
        df = spark.createDataFrame(rows)
        df.coalesce(1).write.mode("overwrite").option("header", "true").csv(filepath)
        print(f"✅ Merchants saved directly to S3 → {filepath}")
        return filepath



# TRANSACTION GENERATOR WITH FRAUD INJECTION


class TransactionGenerator:
    def __init__(
        self,
        customers: List[CustomerProfile],
        merchants: List[MerchantProfile],
        fraud_rate: float = 0.08   
    ):
        self.customers    = customers
        self.merchants    = merchants
        self.fraud_rate   = fraud_rate
        self.transactions: List[Transaction] = []
        self.customer_map = {c.customer_id: c for c in customers}
        self.merchant_map = {m.merchant_id: m for m in merchants}
        self.last_customer_location: dict = {}
        self.last_customer_time: dict     = {}

    def _haversine_km(self, lat1: float, lon1: float, lat2: float, lon2: float) -> float:
        R = 6371  
        phi1, phi2 = math.radians(lat1), math.radians(lat2)
        dphi  = math.radians(lat2 - lat1)
        dlam  = math.radians(lon2 - lon1)
        a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
        return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    def _generate_normal_transaction(self, customer: CustomerProfile, merchant: MerchantProfile, ts: datetime) -> Transaction:
        city_data = random.choice(INDIAN_CITIES)
        amount = round(abs(random.gauss(customer.typical_transaction_amount, customer.typical_transaction_amount * 0.3)), 2)
        amount = max(10.0, min(amount, 500000.0))

        return Transaction(
            transaction_id       = f"TXN_{str(uuid.uuid4()).replace('-','').upper()[:16]}",
            customer_id          = customer.customer_id,
            merchant_id          = merchant.merchant_id,
            amount               = amount,
            currency             = "INR",
            timestamp            = ts.isoformat(),
            transaction_date     = ts.date().isoformat(),
            location_city        = city_data[0],
            location_state       = city_data[1],
            location_country     = "IN",
            latitude             = city_data[2] + random.gauss(0, 0.02),
            longitude            = city_data[3] + random.gauss(0, 0.02),
            ip_address           = fake.ipv4_private(),
            device_id            = random.choice(customer.registered_devices),
            device_type          = random.choice(DEVICE_TYPES[:3]),
            payment_method       = customer.preferred_payment,
            transaction_status   = random.choices(["SUCCESS","FAILED","PENDING"], weights=[88, 8, 4])[0],
            merchant_category    = merchant.merchant_category,
            is_foreign_transaction=False,
            failed_attempt_count = random.choices([0,1,2], weights=[90,7,3])[0],
            response_code        = "00",  
            hour_of_day          = ts.hour,
            day_of_week          = ts.weekday(),
            is_weekend           = ts.weekday() >= 5,
            is_night_transaction = ts.hour < 5 or ts.hour >= 23,
            is_fraud             = False,
            fraud_type           = None,
            fraud_score          = round(random.uniform(0.0, 0.15), 4),
        )

    def _inject_velocity_fraud(self, customer: CustomerProfile, base_ts: datetime) -> List[Transaction]:
        txns = []
        merchant = random.choice(self.merchants)
        for i in range(random.randint(20, 35)):
            ts = base_ts + timedelta(minutes=i * 0.8)   
            txn = self._generate_normal_transaction(customer, merchant, ts)
            txn.transaction_id   = f"TXN_VEL_{str(uuid.uuid4())[:12].upper()}"
            txn.amount           = round(random.uniform(50, 500), 2)
            txn.is_fraud         = True
            txn.fraud_type       = "VELOCITY_FRAUD"
            txn.fraud_score      = round(random.uniform(0.75, 0.98), 4)
            txn.failed_attempt_count = 0
            txns.append(txn)
        return txns

    def _inject_impossible_travel(self, customer: CustomerProfile, base_ts: datetime) -> List[Transaction]:
        txns = []
        domestic_merchant  = random.choice(self.merchants)
        foreign_loc        = random.choice(FOREIGN_LOCATIONS)

        txn1 = self._generate_normal_transaction(customer, domestic_merchant, base_ts)
        txn1.is_fraud     = False
        txn1.fraud_score  = 0.05
        txns.append(txn1)

        ts2 = base_ts + timedelta(minutes=20)
        txn2 = self._generate_normal_transaction(customer, domestic_merchant, ts2)
        txn2.transaction_id        = f"TXN_TRAVEL_{str(uuid.uuid4())[:12].upper()}"
        txn2.location_city         = foreign_loc[0]
        txn2.location_state        = foreign_loc[1]
        txn2.location_country      = foreign_loc[2]
        txn2.latitude              = foreign_loc[3]
        txn2.longitude             = foreign_loc[4]
        txn2.is_foreign_transaction= True
        txn2.currency              = random.choice(["USD","EUR","GBP"])
        txn2.amount                = round(random.uniform(5000, 50000), 2)
        txn2.ip_address            = fake.ipv4_public()
        txn2.is_fraud              = True
        txn2.fraud_type            = "IMPOSSIBLE_TRAVEL"
        txn2.fraud_score           = round(random.uniform(0.85, 0.99), 4)
        txn2.device_id             = str(uuid.uuid4())[:12]  
        txns.append(txn2)
        return txns

    def _inject_aml_structuring(self, customer: CustomerProfile, base_ts: datetime) -> List[Transaction]:
        txns = []
        structured_amounts = [round(random.uniform(9000, 9999), 2) for _ in range(random.randint(4, 8))]

        for i, amount in enumerate(structured_amounts):
            ts = base_ts + timedelta(hours=i * 2)
            merchant = random.choice([m for m in self.merchants if m.merchant_category in ["MONEY_TRANSFER", "CRYPTO_EXCHANGE"]] or self.merchants[:5])
            txn = self._generate_normal_transaction(customer, merchant, ts)
            txn.transaction_id  = f"TXN_AML_{str(uuid.uuid4())[:12].upper()}"
            txn.amount          = amount
            txn.payment_method  = random.choice(["NETBANKING", "WALLET"])
            txn.is_fraud        = True
            txn.fraud_type      = "AML_STRUCTURING"
            txn.fraud_score     = round(random.uniform(0.70, 0.90), 4)
            txns.append(txn)
        return txns

    def _inject_device_anomaly(self, customer: CustomerProfile, base_ts: datetime) -> List[Transaction]:
        merchant = random.choice([m for m in self.merchants if m.is_high_risk_category] or self.merchants[:5])
        txn = self._generate_normal_transaction(customer, merchant, base_ts)
        txn.transaction_id   = f"TXN_DEV_{str(uuid.uuid4())[:12].upper()}"
        txn.device_id        = str(uuid.uuid4())[:12]  
        txn.ip_address       = fake.ipv4_public()      
        txn.amount           = round(random.uniform(10000, 100000), 2)
        txn.merchant_category= "ELECTRONICS"
        txn.is_fraud         = True
        txn.fraud_type       = "DEVICE_ANOMALY"
        txn.fraud_score      = round(random.uniform(0.65, 0.88), 4)
        return [txn]

    def generate(self, num_transactions: int = 50000, start_date: datetime = None, end_date: datetime = None) -> List[Transaction]:
        if start_date is None: start_date = datetime.now() - timedelta(days=30)
        if end_date is None: end_date = datetime.now()

        normal_count = int(num_transactions * (1 - self.fraud_rate))
        fraud_customers = [c for c in self.customers if c.is_synthetic_fraud_customer]

        print(f"  Generating {normal_count:,} normal transactions...")
        for _ in range(normal_count):
            customer = random.choice(self.customers)
            merchant = random.choice(self.merchants)
            ts = start_date + timedelta(seconds=random.randint(0, int((end_date - start_date).total_seconds())))
            self.transactions.append(self._generate_normal_transaction(customer, merchant, ts))

        print(f"  Injecting fraud patterns...")
        injectors = [
            (self._inject_velocity_fraud,    "VELOCITY_FRAUD",    0.25),
            (self._inject_impossible_travel, "IMPOSSIBLE_TRAVEL", 0.20),
            (self._inject_aml_structuring,   "AML_STRUCTURING",   0.20),
            (self._inject_device_anomaly,    "DEVICE_ANOMALY",    0.35),
        ]

        for fraud_customer in fraud_customers:
            injector_fn, fraud_name, weight = random.choices(injectors, weights=[w for _, _, w in injectors])[0]
            ts = start_date + timedelta(seconds=random.randint(0, int((end_date - start_date).total_seconds())))
            fraud_txns = injector_fn(fraud_customer, ts)
            self.transactions.extend(fraud_txns)

        random.shuffle(self.transactions)

        fraud_count = sum(1 for t in self.transactions if t.is_fraud)
        print(f"✅ Generated {len(self.transactions):,} total transactions")
        print(f"   ├─ Normal:    {len(self.transactions) - fraud_count:,}")
        print(f"   ├─ Fraud:     {fraud_count:,}")
        print(f"   └─ Fraud rate: {fraud_count/len(self.transactions)*100:.2f}%")
        return self.transactions

    def save_to_parquet(self, output_path: str, spark=None):
        if spark is None:
            raise ValueError("SparkSession required for Parquet output")

        rows = [asdict(t) for t in self.transactions]
        df = spark.createDataFrame(rows)
        (
            df.repartition(10, "transaction_date")
              .write
              .mode("overwrite")
              .partitionBy("transaction_date")
              .parquet(output_path)
        )
        print(f"✅ Transactions saved directly to S3 as Parquet → {output_path}")



# DATABRICKS ORCHESTRATED GENERATION SCRIPT


def generate_full_dataset(
    s3_output_base: str,
    spark_session,
    num_customers:    int   = 1000,
    num_merchants:    int   = 200,
    num_transactions: int   = 50000,
    days_history:     int   = 30,
):
    print("=" * 60)
    print("  FINTECH FRAUD DETECTION — DATA GENERATION (DATABRICKS TO S3)")
    print("=" * 60)

    # Customers 
    print("\n[1/3] Generating customer profiles...")
    cust_gen   = CustomerGenerator(num_customers)
    customers  = cust_gen.generate()
    cust_gen.save_to_s3(f"{s3_output_base}/landing/customers", spark_session)

    # Merchants 
    print("\n[2/3] Generating merchant profiles...")
    merch_gen  = MerchantGenerator(num_merchants)
    merchants  = merch_gen.generate()
    merch_gen.save_to_s3(f"{s3_output_base}/landing/merchants", spark_session)

    # Transactions 
    print("\n[3/3] Generating transactions with fraud injection...")
    start_date = datetime.now() - timedelta(days=days_history)
    txn_gen    = TransactionGenerator(customers, merchants, fraud_rate=0.08)
    transactions = txn_gen.generate(
        num_transactions=num_transactions,
        start_date=start_date,
        end_date=datetime.now()
    )
    
    # Save directly to S3 natively as Parquet
    txn_gen.save_to_parquet(
        f"{s3_output_base}/landing/transactions",
        spark=spark_session
    )

    print("\n" + "=" * 60)
    print("  DATA GENERATION & S3 UPLOAD COMPLETE")
    print(f"  Target: {s3_output_base}")
    print("=" * 60)

    return customers, merchants, transactions

# Databricks Execution Trigger
if __name__ == "__main__":
    
    S3_BUCKET_PATH = "s3://fintech-fraud-detection-lake"
    
    
    generate_full_dataset(
        s3_output_base=S3_BUCKET_PATH,
        spark_session=spark 
    )